<a href="https://colab.research.google.com/github/LiamCarter2000/Clinical-Data-Related-Projects/blob/main/3%2B3_Simulations_orig_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports

In [20]:
import pandas as pd
import numpy as np
import math
import heapq
from typing_extensions import final
from tabulate import tabulate
from collections import defaultdict

#Data Processing

In [21]:
df = pd.read_csv("Decision Grid.csv", usecols=[1, 2, 3, 4, 5, 6, 7, 8])

df.columns = ['Treated', 'DLT', 'Pending', 'State', 'Orig_33_Action', 'IQ_33_Action', 'Orig_6_Action', 'IQ_6_Action']

decision_dict_IQ_33 = df.set_index(['Treated', 'DLT', 'Pending', 'State'])['IQ_33_Action'].to_dict()
decision_dict_33 = df.set_index(['Treated', 'DLT', 'Pending', 'State'])['Orig_33_Action'].to_dict()
decision_dict_IQ_6 = df.set_index(['Treated', 'DLT', 'Pending', 'State'])['IQ_6_Action'].to_dict()
decision_dict_6 = df.set_index(['Treated', 'DLT', 'Pending', 'State'])['Orig_6_Action'].to_dict()

In [22]:
# visualization of the processed data
print(df.head(15))

    Treated  DLT  Pending   State Orig_33_Action IQ_33_Action Orig_6_Action  \
0         0    0        0    open           Stay         Stay          Stay   
1         0    0        0  closed           Stay         Stay          Stay   
2         1    0        0    open           Stay         Stay          Stay   
3         1    0        0  closed           Stay         Stay          Stay   
4         1    1        0    open           Stay         Stay          Stay   
5         1    1        0  closed           Stay         Stay          Stay   
6         1    0        1    open           Stay         Stay          Stay   
7         1    0        1  closed           Stay         Stay          Stay   
8         2    0        0    open           Stay         Stay          Stay   
9         2    0        0  closed           Stay         Stay          Stay   
10        2    1        0    open           Stay         Stay          Stay   
11        2    1        0  closed           Stay    

In [23]:
print(f"Decision for Row index 29 3+3 (3,0,1,open): {decision_dict_33.get((3, 0, 1, 'open'))}")
print(f"Decision for Row index 29 IQ 3+3 (3,0,1,open): {decision_dict_IQ_33.get((3, 0, 1, 'open'))}")
print(f"Decision for Row index 29 rolling 6 (3,0,1,open): {decision_dict_IQ_6.get((3, 0, 1, 'open'))}")
print(f"Decision for Row index 29 IQ rolling 6 (3,0,1,open): {decision_dict_6.get((3, 0, 1, 'open'))}")


Decision for Row index 29 3+3 (3,0,1,open): Suspend
Decision for Row index 29 IQ 3+3 (3,0,1,open): Stay
Decision for Row index 29 rolling 6 (3,0,1,open): Stay
Decision for Row index 29 IQ rolling 6 (3,0,1,open): Stay


#Global Variables

Data is processed into four python dictionaries: one for each of the methodologies for which the simulation will run (original 3+3, IQ 3+3, original rolling 6, IQ rolling 6). The values in each of the columns of these methodologies are as follows.

 Defines what dose level the next patient should be treated at, based on the current number of patients being treated, the number of reported DLTs, and the number of results pending at the current dose level.

Possible Dispositions:

1. Up          = start testing patients at the next higher dose level
2. Down     = start testing patients at the next lower dose level
3. Stay       = keep testing patients at the current dose level
4. Suspend = wait for additional testing to finish at current dose
                      level, before starting a test on another patient.
5. MTD       = current dose is the Maximum Tolerated Dose (MTD)


The value of having this as a python dictionary is that all values can be looked up in place for a much faster run time. So if we want to know what to do for IQ 3+3 when we have Treated = 3, DLT = 0, Pending = 1, and State = open, we can just query the corresponding dictionary with "decision_dict_IQ_33.get((3, 0, 1, 'open'))" to get "Stay"

All that is left is to simulate the actions of the dispositions while also considering possible dropout of people that develop DLT.




In [46]:
Starting_Dose_Level = 2
Lowest_Possible_Dose_Level = 1
Highest_Possible_Dose_Level = 5

#(maximum duration interested patients will wait for a slot)
Waitlist_Time = 0
#(duration of DLT evaluation period)
Course_Length = 28
#(percent who will fail the screening process)
Screen_Failure_Prob = 0.30
#(percent inevaluable.  Actual inevaluable rate will be lower due to competing DLT risk)
Inevaluable_Prob = 0.20


# DLT Probability Function (defines the probability of a patient having a DLT event, based on dose)
def get_dlt_probability(dose_level, custom_probs=None):
  """
  Returns the DLT probability.
  Defaults to the arctan model unless a custom list is provided.
  """
  if dose_level is None or dose_level <= 0:
    return 0.0

  # If a custom list is provided (e.g. [.1, .15, .25, .30, .35, .5])
  if custom_probs is not None:
    idx = dose_level - 1
    # Return the value at the index, or the last value if dose exceeds list length
    if idx < len(custom_probs):
      return custom_probs[idx]
    else:
      return custom_probs[-1]

  # Default: Arctan Formula
  val = 0.5 + (math.atan(0.2 * math.pi * (dose_level - 8.5)) / math.pi)
  return max(0.0, min(1.0, val))


#Patient Inter-Arrival Time (days between arrivals)
def sample_interarrival(rng):
    # return math.floor(rng.exponential(10)) # Mean of x days
    return rng.exponential(10)

#Screening Duration Distribution (length of screening process)
def sample_screening_duration(rng):
    # beta(0, 28, 1, 1) is a uniform distribution between 0 and 28
    return math.floor(rng.uniform(0, Course_Length + 1))

#Time Until DLT Event (distribution of times before the patient will have the DLT event)
def sample_dlt_timing(rng):
    # beta(0, CourseLength, 1.5, 1) skewed toward the end of the cycle
    return math.floor(rng.beta(1.5, 1) * Course_Length + 1)

#Time Until Inevaluable (distribution of times before the patient will become inevaluable)
def sample_inevaluable_timing(rng):
    # beta(0,CourseLength,1,1) is a Uniform(0, 28) distribution
    return math.floor(rng.beta(1, 1) * Course_Length + 1)


#Patient Class

In [25]:
class Patient:
  def __init__(self, arrival_time, dose, ID, rng, course_length = Course_Length, custom_dlt_probs = None):
    self.ID = ID
    self.arrival_time = arrival_time
    self.dose = dose
    self.course_length = course_length #length of study
    self.rng = rng
    self.custom_dlt_probs = custom_dlt_probs
    self.started_treatment = False
    self.treatment_start_time = None

    self.screening_outcome, self.days_to_screening_outcome = self.determine_screening_outcome(rng)

    self.started_screening = False
    self.screening_start_time = None
    self.screening_end_time = None

    self.started_treatment = False
    self.treatment_start_time = None
    self.resolution_time = None
    # will be determined in start treatment, not now because dose might change.
    self.treatment_outcome = None
    self.days_to_treatment_outcome = None


  def determine_screening_outcome(self, rng):
    screen_duration = sample_screening_duration(rng)
    if rng.random() < Screen_Failure_Prob:
      return "fail", screen_duration
    else:
      return "pass", screen_duration

  def start_screening(self, current_clock_time):
    self.started_screening = True
    self.screening_start_time = current_clock_time
    self.screening_end_time = self.screening_start_time + self.days_to_screening_outcome


  # returns "dlt", "ineval", or "pass", as well as a duration til it happens
  def determine_patient_outcome(self, rng):
    dlt_prob = get_dlt_probability(self.dose, self.custom_dlt_probs)
    ineval_prob = Inevaluable_Prob

    time_to_ineval = -1
    time_to_dlt = -1

    if rng.random() < ineval_prob:
      time_to_ineval = sample_inevaluable_timing(rng)

    if rng.random() < dlt_prob:
      time_to_dlt = sample_dlt_timing(rng)

    # if these both happen, we choose the one that will happen first in the timeline
    if time_to_ineval > -1 and time_to_dlt > -1:
      if time_to_ineval < time_to_dlt:
        return "ineval", time_to_ineval
      else:
        return "dlt", time_to_dlt
    elif time_to_ineval > -1:
      return "ineval", time_to_ineval
    elif time_to_dlt > -1:
      return "dlt", time_to_dlt
    else:
      return "pass", self.course_length

  # current_clock_time will be an int representation of days since trial
  # resolution_time is the day they pass, get dlt, or become ineval
  def start_treatment(self, current_clock_time):
    self.treatment_outcome, self.days_to_treatment_outcome = self.determine_patient_outcome(self.rng)
    self.started_treatment = True
    self.treatment_start_time = current_clock_time
    self.resolution_time = self.treatment_start_time + self.days_to_treatment_outcome


#Simulation Code
- running this runs one simulation

## helpers

In [26]:
def archive(treated_list, pending_list):
  treated_list.extend(pending_list)
  pending_list.clear()

def get_decision_dict(trial_type):
  if trial_type == 1:
    return decision_dict_33
  elif trial_type == 2:
    return decision_dict_IQ_33
  elif trial_type == 3:
    return decision_dict_6
  elif trial_type == 4:
    return decision_dict_IQ_6
  else:
    print("Invalid trial type: Please choose a value 1-4")
    return None

def handle_patient_arrival(next_arrival_time, current_clock_time,
                           dose_level, patientID, rng, custom_dlt_probs,
                           screening_list, patient_states, overlooked_patients,
                           current_trial_event, decision_dict):

  while next_arrival_time <= current_clock_time:
    state = patient_states[dose_level]
    current_trial_event = decision_dict.get((
      state['treated'],
      state['DLT'],
      state['pending'],
      state['AboveLevelState']), "Error")

    if current_trial_event == "Stay":
      new_patient = Patient(current_clock_time, dose_level, patientID, rng, custom_dlt_probs= custom_dlt_probs)
      patientID += 1
      new_patient.start_screening(current_clock_time)
      screening_list.append(new_patient)

      # If they are screening, they take up a spot in the study
      patient_states[dose_level]["pending"] += 1
      patient_states[dose_level]["treated"] += 1
      next_arrival_time += sample_interarrival(rng)
    else:
      overlooked_patients += 1
      next_arrival_time += sample_interarrival(rng)

    if next_arrival_time > 0 or current_trial_event != "Stay":
      break

  return next_arrival_time, patientID, overlooked_patients, current_trial_event

def resolve_patient_outcomes(current_clock_time, dose_level, screening_list, pending_list,
                             patient_states, treated_list, total_inevals, decision_dict):
  state = patient_states[dose_level]

  # resolve ongoing treatment
  for patient in pending_list[:]:
    if patient.resolution_time == current_clock_time:
      pending_list.remove(patient)
      state["pending"] -= 1
      if patient.treatment_outcome == "ineval":
        state["treated"] -= 1
        total_inevals += 1
      elif patient.treatment_outcome == "dlt":
        state["DLT"] += 1
      treated_list.append(patient)

  # resolve ongoing screening
  while screening_list and screening_list[0].screening_end_time <= current_clock_time:
    patient = screening_list[0]

    # If they fail screening, they leave the study immediately
    if patient.screening_outcome == "fail":
      screening_list.pop(0)
      state["pending"] -= 1
      state["treated"] -= 1
      continue

    screening_list.pop(0)
    # Re-verify dose
    patient.dose = dose_level
    patient.start_treatment(current_clock_time)
    pending_list.append(patient)

  return total_inevals




def remove_screening_from_dose(dose, screening_list, patient_states):
  # Removes the 'reservation' for screening patients from a dose level.
  for patient in screening_list:
      patient_states[dose]["treated"] -= 1
      patient_states[dose]["pending"] -= 1

def add_screening_to_dose(dose, screening_list, patient_states):
  # Assigns screening patients to a new dose level and updates counts.
  for patient in screening_list:
      patient.dose = dose
      patient_states[dose]["treated"] += 1
      patient_states[dose]["pending"] += 1

def finalize_pending_at_dose(dose, pending_list, patient_states):
  # Resolves outcomes for patients currently in treatment at a dose being exited.
  for patient in pending_list:
    patient_states[dose]["pending"] -= 1
    if patient.treatment_outcome == "dlt":
      patient_states[dose]["DLT"] += 1
    elif patient.treatment_outcome == "ineval":
      patient_states[dose]["treated"] -= 1


## event based sim

In [48]:
def simulation(trial_type, custom_dlt_probs=None, seed=None):
  rng = np.random.default_rng(seed)
  decision_dict = get_decision_dict(trial_type)
  event_queue = []
  treated_list = []
  screening_list = []
  pending_list = []
  current_clock_time = 0.0

  ## Trial State
  trial_status = True
  # options are Stay, Up, Down, Suspend, MTD
  current_trial_event = "Stay"
  dose_level = Starting_Dose_Level
  max_dose_level = Highest_Possible_Dose_Level
  min_dose_level = Lowest_Possible_Dose_Level
  final_dose_level = None

  patientID = 1
  total_inevals = 0
  overlooked_patients = 0

  # Event Types
  ARRIVAL = 1
  SCREENING_END = 2
  TREATMENT_END = 3

  patient_states = {}
  for d in range(min_dose_level, max_dose_level + 1):
    patient_states[d] = {
      "treated": 0,
      "DLT": 0,
      "pending": 0,
      "AboveLevelState": "closed" if d == max_dose_level else "open"
    }

  # Helper to get the event_type
  def get_event():
    return decision_dict.get((
      patient_states[dose_level]['treated'],
      patient_states[dose_level]['DLT'],
      patient_states[dose_level]['pending'],
      patient_states[dose_level]['AboveLevelState']), "Error")

  # Start the trial with the first arrival
  first_arrival = sample_interarrival(rng)
  # heap called event queue has the arrival time, event type,
  heapq.heappush(event_queue, (first_arrival, ARRIVAL, None))

  while event_queue and trial_status:
    # jump the clock
    event_time, event_type, patient = heapq.heappop(event_queue)
    current_clock_time = event_time

    # patient arrives
    if event_type == ARRIVAL:
      # Schedule the next arrival regardless of current trial state
      state = patient_states[dose_level]
      next_arrival_time = current_clock_time + sample_interarrival(rng)
      heapq.heappush(event_queue, (next_arrival_time, ARRIVAL, None))

      if current_trial_event == "Stay":
        new_patient = Patient(current_clock_time, dose_level, patientID, rng, custom_dlt_probs= custom_dlt_probs)
        patientID += 1
        new_patient.start_screening(current_clock_time)
        screening_list.append(new_patient)
        heapq.heappush(event_queue, (new_patient.screening_end_time, SCREENING_END, new_patient))

        # If they are screening, they take up a spot in the study
        state["pending"] += 1
        state["treated"] += 1

      else:
        overlooked_patients += 1

    # patient finishes screening or inevals
    elif event_type == SCREENING_END:
      # Important: Check if patient is still in the screening_list (hasn't been cleared)
      state = patient_states[dose_level]
      if patient in screening_list:
        screening_list.remove(patient)
        if patient.screening_outcome == "fail":
          state["pending"] -= 1
          state["treated"] -= 1
        elif patient.screening_outcome == "pass":
          # Re-verify dose
          patient.dose = dose_level
          patient.start_treatment(current_clock_time)
          pending_list.append(patient)
          heapq.heappush(event_queue, (patient.resolution_time, TREATMENT_END, patient))

    # patient finishes treatment or dlt or inevals
    elif event_type == TREATMENT_END:
      state = patient_states[dose_level]
      if patient in pending_list:
        pending_list.remove(patient)
        state["pending"] -= 1
        if patient.treatment_outcome == "ineval":
          state["treated"] -= 1
          total_inevals += 1
        elif patient.treatment_outcome == "dlt":
          state["DLT"] += 1
        treated_list.append(patient)

    # update event type based on above factors
    current_trial_event = get_event()

    if current_trial_event == "Up":
      old_dose_level = dose_level
      dose_level += 1
      new_state = "closed" if dose_level == max_dose_level else "open"
      patient_states[dose_level]["AboveLevelState"] = new_state

      remove_screening_from_dose(old_dose_level, screening_list, patient_states)
      finalize_pending_at_dose(old_dose_level, pending_list, patient_states)
      add_screening_to_dose(dose_level, screening_list, patient_states)

      archive(treated_list, pending_list)

    elif current_trial_event == "Down":
      remove_screening_from_dose(dose_level, screening_list, patient_states)
      finalize_pending_at_dose(dose_level, pending_list, patient_states)

      while True:
        if dose_level == min_dose_level:
          final_dose_level = dose_level -1
          trial_status = False
          break

        dose_level -= 1
        patient_states[dose_level]["AboveLevelState"] = "closed"
        current_trial_event = get_event()

        if current_trial_event == "Down":
          # Level is still too toxic, down again
          continue
        elif current_trial_event == "MTD":
          final_dose_level = dose_level
          trial_status = False
          break
        else:
          # NOW we can safely add the screening patients
          if current_trial_event != "Error":
            add_screening_to_dose(dose_level, screening_list, patient_states)
            patient_needed = True
          else:
            trial_status = False
          break

      archive(treated_list, pending_list)

    # if current_trial_event == "Suspend" or current_trial_event == "Stay":
      # nothing happens here

    if current_trial_event == "MTD":
      archive(treated_list, pending_list)
      final_dose_level = dose_level
      trial_status = False

    if current_trial_event == "Error":
      archive(treated_list, pending_list)
      final_dose_level = dose_level

      if dose_level == min_dose_level:
        result_summary = "Terminated: Too Toxic at Lowest Level"
      else:
        result_summary = "Terminated: Error/Inconclusive"

      print(f"Error in chart reached at seed {seed}")
      print(f"Reason: {result_summary}")
      end_on_error = True
      trial_status = False

    current_trial_event = get_event()

  if final_dose_level is None:
    print("trial timeout")
    final_dose_level = dose_level

  # returns list of all treated/dlt patients, days_taken, final_dose_level, total_inevals
  return treated_list, current_clock_time, final_dose_level, total_inevals



#Debug code for simulation
- slightly outdated to the main code

## Helper methods

In [71]:
def debug_log(log, day, event, action, dose, states):
  return f"Day: {day:<7} | Event: {event:<10} | {action:<77} | Dose: {dose:<4} | States: {states} \n"

def get_decision_dict_debug(trial_type, event_log):
  if trial_type == 1:
    event_log += "Beginning 3+3 Simulation\n"
    return decision_dict_33
  elif trial_type == 2:
    event_log += "Beginning 3+3 Simulation\n"
    return decision_dict_IQ_33
  elif trial_type == 3:
    event_log += "Beginning 3+3 Simulation\n"
    return decision_dict_6
  elif trial_type == 4:
    event_log += "Beginning 3+3 Simulation\n"
    return decision_dict_IQ_6
  else:
    print("Invalid trial type: Please choose a value 1-4")
    return None

#Handles the state transfer for patients when the dose level changes
def migrate_patients_on_dose_change_debug_up(
    old_dose, new_dose, screening_list, pending_list, patient_states,
    current_clock_time, current_trial_event, event_log):
  # Migrate screening patients
  screen_string = ""
  for patient in screening_list:
    patient_states[old_dose]["treated"] -= 1
    patient_states[old_dose]["pending"] -= 1
    patient.dose = new_dose
    patient_states[new_dose]["treated"] += 1
    patient_states[new_dose]["pending"] += 1
    screen_string += f"{patient.ID}, "

  ids = [p.ID for p in pending_list]
  event_log += debug_log(event_log, round(current_clock_time, 2), current_trial_event, f"Patient {ids} archived. Dose goes up. Patients: {screen_string}migrated from screening", new_dose, patient_states[new_dose])

  # Resolve outcomes for those already in treatment
  for patient in pending_list:
    patient_states[old_dose]["pending"] -= 1
    event_log += debug_log(event_log, round(current_clock_time, 2), current_trial_event, f"Patient {patient.ID} will resolve to {patient.treatment_outcome} on day {patient.resolution_time}", new_dose, patient_states[new_dose])
    if patient.treatment_outcome == "dlt":
      patient_states[old_dose]["DLT"] += 1
    elif patient.treatment_outcome == "ineval":
      patient_states[old_dose]["treated"] -= 1

  return event_log

def migrate_patients_on_dose_change_debug_down(
    old_dose, new_dose, screening_list, pending_list, patient_states,
    current_clock_time, current_trial_event, event_log):
  # Migrate screening patients
  screen_string = ""
  for patient in screening_list:
    patient_states[old_dose]["treated"] -= 1
    patient_states[old_dose]["pending"] -= 1
    patient.dose = new_dose
    patient_states[new_dose]["treated"] += 1
    patient_states[new_dose]["pending"] += 1
    screen_string += f"{patient.ID}, "

  ids = [p.ID for p in pending_list]
  event_log += debug_log(event_log, round(current_clock_time, 2), current_trial_event, f"Patient {ids} archived. Dose goes down. Patients: {screen_string}migrated from screening", new_dose, patient_states[new_dose])

  return event_log





##new method

In [61]:
# New code
def simulation_with_debug(trial_type, custom_dlt_probs=None, seed=None):
  # no actual seed on debug unless you want
  rng = None
  if seed == None:
    ss = np.random.SeedSequence(seed)
    actual_seed = ss.entropy
    #print(f"Running simulation with seed: {actual_seed}")
    rng = np.random.default_rng(ss)
    seed = actual_seed
  else:
    rng = np.random.default_rng(seed)

  # to keep track of patinets
  decision_dict = get_decision_dict(trial_type)
  event_queue = []
  treated_list = []
  screening_list = []
  pending_list = []
  current_clock_time = 0.0

  # debug specifig variables
  event_log = ""
  end_on_error = False
  num_consented_patients = 0
  num_started_treatment = 0

  def get_day():
    return round(current_clock_time, 2)

  ## Trial State
  trial_status = True
  # options are Stay, Up, Down, Suspend, MTD
  current_trial_event = "Stay"
  dose_level = Starting_Dose_Level
  max_dose_level = Highest_Possible_Dose_Level
  min_dose_level = Lowest_Possible_Dose_Level
  final_dose_level = None

  patientID = 1
  total_inevals = 0
  overlooked_patients = 0

  # Event Types
  ARRIVAL = 1
  SCREENING_END = 2
  TREATMENT_END = 3

  patient_states = {}
  for d in range(min_dose_level, max_dose_level + 1):
    patient_states[d] = {
      "treated": 0,
      "DLT": 0,
      "pending": 0,
      "AboveLevelState": "closed" if d == max_dose_level else "open"
    }

  # Helper to get the event_type
  def get_event():
    return decision_dict.get((
      patient_states[dose_level]['treated'],
      patient_states[dose_level]['DLT'],
      patient_states[dose_level]['pending'],
      patient_states[dose_level]['AboveLevelState']), "Error")

  # Start the trial with the first arrival
  first_arrival = sample_interarrival(rng)
  # heap called event queue has the arrival time, event type,
  heapq.heappush(event_queue, (first_arrival, ARRIVAL, None))

  while event_queue and trial_status:
    # jump the clock
    event_time, event_type, patient = heapq.heappop(event_queue)
    current_clock_time = event_time

    # patient arrives
    if event_type == ARRIVAL:
      # Schedule the next arrival regardless of current trial state
      state = patient_states[dose_level]
      next_arrival_time = current_clock_time + sample_interarrival(rng)
      heapq.heappush(event_queue, (next_arrival_time, ARRIVAL, None))

      if current_trial_event == "Stay":
        new_patient = Patient(current_clock_time, dose_level, patientID, rng, custom_dlt_probs= custom_dlt_probs)
        patientID += 1
        new_patient.start_screening(current_clock_time)
        screening_list.append(new_patient)
        num_consented_patients += 1
        heapq.heappush(event_queue, (new_patient.screening_end_time, SCREENING_END, new_patient))

        # If they are screening, they take up a spot in the study
        state["pending"] += 1
        state["treated"] += 1
        event_log += debug_log(event_log, get_day(), current_trial_event, f"Adding new patient {new_patient.ID} to screening.", dose_level, patient_states[dose_level])

      else:
        overlooked_patients += 1
        # event_log += debug_log(event_log, get_day(), current_trial_event, f"Turned away possible patient, time to next patient = {days_before_next_patient}", dose_level, patient_states[dose_level])

    # patient finishes screening or inevals
    elif event_type == SCREENING_END:
      # Important: Check if patient is still in the screening_list (hasn't been cleared)
      state = patient_states[dose_level]
      if patient in screening_list:
        screening_list.remove(patient)
        if patient.screening_outcome == "fail":
          state["pending"] -= 1
          state["treated"] -= 1
          event_log += debug_log(event_log, get_day(), current_trial_event, f"Patient {patient.ID} failed screening.",dose_level, patient_states[dose_level])
        elif patient.screening_outcome == "pass":
          # Re-verify dose
          patient.dose = dose_level
          patient.start_treatment(current_clock_time)
          num_started_treatment += 1
          pending_list.append(patient)
          heapq.heappush(event_queue, (patient.resolution_time, TREATMENT_END, patient))
          event_log += debug_log(event_log, get_day(), current_trial_event, f"Patient {patient.ID} starts treatment, at dose level {patient.dose}.", dose_level, patient_states[dose_level])

    # patient finishes treatment or dlt or inevals
    elif event_type == TREATMENT_END:
      state = patient_states[dose_level]
      if patient in pending_list:
        pending_list.remove(patient)
        state["pending"] -= 1
        if patient.treatment_outcome == "pass":
          event_log += debug_log(event_log, get_day(), current_trial_event, f"Patient {patient.ID} passes treatment.", dose_level, state)
        if patient.treatment_outcome == "ineval":
          state["treated"] -= 1
          total_inevals += 1
          event_log += debug_log(event_log, get_day(), current_trial_event, f"Removing patient {patient.ID} for ineval.", dose_level, state)
        elif patient.treatment_outcome == "dlt":
          state["DLT"] += 1
          event_log += debug_log(event_log, get_day(), current_trial_event, f"Removing patient {patient.ID} for ineval.", dose_level, state)
        treated_list.append(patient)

    # update event type based on above factors
    current_trial_event = get_event()

    if current_trial_event == "Up":
      old_dose_level = dose_level
      dose_level += 1
      new_state = "closed" if dose_level == max_dose_level else "open"
      patient_states[dose_level]["AboveLevelState"] = new_state

      #migrating the screening patients to the next dose
      event_log = migrate_patients_on_dose_change_debug_up(
        old_dose_level, dose_level, screening_list, pending_list, patient_states,
        current_clock_time, current_trial_event, event_log)

      archive(treated_list, pending_list)

    elif current_trial_event == "Down":
      remove_screening_from_dose(dose_level, screening_list, patient_states)
      finalize_pending_at_dose(dose_level, pending_list, patient_states)

      p_ids = [p.ID for p in pending_list]
      s_ids = [p.ID for p in screening_list]

      while True:
        if dose_level == min_dose_level:
          final_dose_level = dose_level -1
          trial_status = False
          break

        dose_level -= 1
        patient_states[dose_level]["AboveLevelState"] = "closed"
        current_trial_event = get_event()

        if current_trial_event == "Down":
          # Level is still too toxic, down again
          continue
        elif current_trial_event == "MTD":
          final_dose_level = dose_level
          trial_status = False
          event_log += debug_log(event_log, get_day(), current_trial_event, f"Decreasing Dose. Moving patients {p_ids} to treated from pending, dose is now MTD.", dose_level, patient_states[dose_level])
          break
        else:
          # NOW we can safely add the screening patients
          if current_trial_event != "Error":
            add_screening_to_dose(dose_level, screening_list, patient_states)
            patient_needed = True
          else:
            trial_status = False
          break

      event_log += debug_log(event_log, get_day(), current_trial_event, f"Patient {p_ids} archived and {s_ids} transferred. Dose goes down to dose level {dose_level}.", dose_level, patient_states[dose_level])
      archive(treated_list, pending_list)

    # if current_trial_event == "Suspend" or current_trial_event == "Stay":
      # nothing happens here

    if current_trial_event == "MTD":
      ids = [p.ID for p in pending_list]
      event_log += debug_log(event_log, get_day(), current_trial_event, f"Moving patients {ids} to treated from pending, dose is MTD.", dose_level, patient_states[dose_level])

      archive(treated_list, pending_list)

      final_dose_level = dose_level
      trial_status = False

    if current_trial_event == "Error":
      ids = [p.ID for p in pending_list]
      event_log += debug_log(event_log, get_day(), current_trial_event, f"Moving patients {ids} to treated from pending, error.", dose_level, patient_states[dose_level])

      archive(treated_list, pending_list)
      final_dose_level = dose_level

      if dose_level == min_dose_level:
        result_summary = "Terminated: Too Toxic at Lowest Level"
      else:
        result_summary = "Terminated: Error/Inconclusive"

      print(f"Error in chart reached at seed {seed}")
      print(f"Reason: {result_summary}")
      end_on_error = True
      trial_status = False

    current_trial_event = get_event()

  if final_dose_level is None:
    print("trial timeout")
    final_dose_level = dose_level

  num_fully_evaluable = len([p for p in treated_list if p.treatment_outcome != "ineval"])
  event_log += f"Number of patients consented to screening: {num_consented_patients}\n"
  event_log += f"Number of patients starting treatment: {num_started_treatment}\n"
  event_log += f"Number of patients fully evaluable: {num_fully_evaluable}\n"
  event_log += f"Final Dose Level: {final_dose_level}\n"
  event_log += f"Total Time Taken: {current_clock_time} days or ~{round(current_clock_time / 30.375 , 2)} months\n"
  # print(f"Overlooked patients: {overlooked_patients}")

  # returns list of all treated/dlt patients, days_taken, final_dose_level, total_inevals
  return treated_list, current_clock_time, final_dose_level, total_inevals, event_log, end_on_error, seed



In [58]:
def run_debug_test(sim_type=1, sdl=2, hpd=5, lpd=1, wt=0, cl=28, sfp=0.3, ip=0.2, seed = None):
  global Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level
  global Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob

  orig = (Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level,
          Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob)

  Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level = sdl, hpd, lpd
  Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob = wt, cl, sfp, ip

  result = simulation_with_debug(sim_type, seed = seed)
  print(f"Seed: {result[6]}\n {result[4]}") # Prints the event_log

  Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level = orig[0], orig[1], orig[2]
  Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob = orig[3], orig[4], orig[5], orig[6]

In [59]:
def run_debug_test_infinite(sim_type=1, reps = 10000, sdl=2, hpd=5, lpd=1, wt=0, cl=28, sfp=0.3, ip=0.2, seed = None):
  global Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level
  global Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob

  orig = (Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level,
          Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob)

  Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level = sdl, hpd, lpd
  Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob = wt, cl, sfp, ip

  for i in range(reps):
    result = simulation_with_debug(sim_type)
    if result[5]:
      print(f"Seed: {result[6]}\n {result[4]}")
      break

  Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level = orig[0], orig[1], orig[2]
  Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob = orig[3], orig[4], orig[5], orig[6]

#Metrics
- This runs x num of simulations and returns the metrics
- Used directly in the run_batch_report dunction

In [32]:
def get_metrics(simulation_type, num_repetitions, custom_dlt_probs, seed):
  metrics, dose_stats = [], defaultdict(lambda: {"dlts": 0, "total": 0})

  # NOW SEEDED
  for i in range(num_repetitions):
    # treated_list now includes ALL patients (total treated)
    treated_list, days_taken, final_dose, total_inevals = simulation(simulation_type, seed = i + seed, custom_dlt_probs= custom_dlt_probs)

    # Identify Evaluable patients (outcome is not 'ineval')
    evaluable_list = [p for p in treated_list if p.treatment_outcome != "ineval"]
    num_treated = len(treated_list)
    num_evaluable = len(evaluable_list)

    num_DLT_above_MTD = 0
    for p in treated_list:
      dose_stats[p.dose]["total"] += 1
      if p.treatment_outcome == "dlt":
        dose_stats[p.dose]["dlts"] += 1
        if p.dose > final_dose: num_DLT_above_MTD += 1

    metrics.append({
      "days_taken": days_taken,
      "final_dose": final_dose,
      "num_treated": num_treated,      # Total count
      "num_evaluable": num_evaluable,  # Completed DLT window
      "num_DLT_above_MTD": num_DLT_above_MTD
    })
  return pd.DataFrame(metrics), dose_stats

def format_theoretical_dlt_col(max_level=5, custom_dlt_probs= None):
  # Formats the 'Ground Truth' column based on either custom list or mathematical model.
  lines = ["NA  NA"]
  for level in range(1, max_level + 1):
    prob = get_dlt_probability(level, custom_dlt_probs)
    rate = prob * 100
    lines.append(f"{str(level).ljust(2)}  {rate:.1f}%")
  return "\n".join(lines)


def bucket_dose(val): return int(val) if val and val > 0 else "NA"

def format_pct(val):
  if val == 0: return "0.0%"
  return "<1.0%" if 0 < val < 1.0 else f"{val:.1f}%"

def get_mtd_breakdown(doses, max_lvl=5):
  bucketed = doses.apply(bucket_dose)
  probs = (bucketed.value_counts() / len(doses) * 100)
  order = ["NA"] + list(range(1, max_lvl + 1))
  return probs.reindex(order, fill_value=0.0).apply(format_pct).to_frame("% MTD At level")

def format_stat_str(ser):
  m, med = round(ser.mean(), 1), round(ser.median(), 1)
  mi, ma = round(ser.min(), 1), round(ser.max(), 1)
  f_mi = int(mi) if mi == int(mi) else mi
  f_ma = int(ma) if ma == int(ma) else ma
  return f"{m}, {med}\n({f_mi}-{f_ma})"

def format_mtd_col(doses, max_lvl=5):
  df = get_mtd_breakdown(doses, max_lvl)
  return "\n".join([f"{str(lvl).ljust(2)}  {row['% MTD At level']}" for lvl, row in df.iterrows()])

In [33]:
#TODO :: need to change the code so it doesn't run on global variables, this is an almalgamation
def run_batch_report(scenarios, sim_types, num_reps, seed=0):
  global Starting_Dose_Level, Highest_Possible_Dose_Level, Lowest_Possible_Dose_Level
  global Waitlist_Time, Course_Length, Screen_Failure_Prob, Inevaluable_Prob

  # 1. Store the "True" Global Defaults before we change anything
  initial_state = {
    "Starting_Dose_Level": Starting_Dose_Level,
    "Highest_Possible_Dose_Level": Highest_Possible_Dose_Level,
    "Lowest_Possible_Dose_Level": Lowest_Possible_Dose_Level,
    "Waitlist_Time": Waitlist_Time,
    "Course_Length": Course_Length,
    "Screen_Failure_Prob": Screen_Failure_Prob,
    "Inevaluable_Prob": Inevaluable_Prob
  }

  try:
    type_map = {1: "3+3", 2: "IQ 3+3", 3: "Rolling 6", 4: "IQ Rolling 6"}
    table_rows, headers = [], []

    for p in scenarios:
      # 2. Update globals for this specific loop iteration
      # Always fallback to initial_state, NOT the current global (which might have been changed by a previous loop)
      Starting_Dose_Level         = p.get("Starting_Dose_Level",         initial_state["Starting_Dose_Level"])
      Highest_Possible_Dose_Level = p.get("Highest_Possible_Dose_Level", initial_state["Highest_Possible_Dose_Level"])
      Lowest_Possible_Dose_Level  = p.get("Lowest_Possible_Dose_Level",  initial_state["Lowest_Possible_Dose_Level"])
      Waitlist_Time               = p.get("Waitlist_Time",               initial_state["Waitlist_Time"])
      Course_Length               = p.get("Course_Length",               initial_state["Course_Length"])
      Screen_Failure_Prob         = p.get("Screen_Failure_Prob",         initial_state["Screen_Failure_Prob"])
      Inevaluable_Prob            = p.get("Inevaluable_Prob",            initial_state["Inevaluable_Prob"])

      custom_dlt_probs = p.get("dlt_probs", None)
      row = {"Variation": p.get("name", "Unnamed")}
      tr_cols, ev_cols, mo_cols, mtd_cols, dlt_sum = {}, {}, {}, {}, []

      for st in sim_types:
        df, _ = get_metrics(st, num_reps, custom_dlt_probs, seed)
        df['months_taken'] = df['days_taken'] / 30.4375
        lbl = type_map.get(st, f"T{st}")

        tr_cols[f"#Treated\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['num_treated'])
        ev_cols[f"#Evaluable\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['num_evaluable'])
        mo_cols[f"#Months\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['months_taken'])

        dlt_sum.append(f"{lbl}: {df['num_DLT_above_MTD'].mean():.2f}")
        mtd_cols[f"{lbl}\n% MTD\nAt level"] = format_mtd_col(df['final_dose'].copy(), Highest_Possible_Dose_Level)

      row.update(tr_cols)
      row.update(ev_cols)
      row.update(mo_cols)
      row["Mean\n#DLTs\nabove\nMTD"] = "\n".join(dlt_sum)
      row.update(mtd_cols)
      row["DLT rate\n(based\non model)"] = format_theoretical_dlt_col(Highest_Possible_Dose_Level, custom_dlt_probs)

      if not headers:
        headers = list(row.keys())
      table_rows.append(list(row.values()))

    print(tabulate(table_rows, headers=headers, tablefmt="grid"))

  finally:
    # 3. RESTORE EVERYTHING back to original dose values/settings
    # This runs even if the code above hits an error.
    Starting_Dose_Level         = initial_state["Starting_Dose_Level"]
    Highest_Possible_Dose_Level = initial_state["Highest_Possible_Dose_Level"]
    Lowest_Possible_Dose_Level  = initial_state["Lowest_Possible_Dose_Level"]
    Waitlist_Time               = initial_state["Waitlist_Time"]
    Course_Length               = initial_state["Course_Length"]
    Screen_Failure_Prob         = initial_state["Screen_Failure_Prob"]
    Inevaluable_Prob            = initial_state["Inevaluable_Prob"]

# Simulation Benchmarking (eTable 3 Comparison)

This code block runs simulations and generates a report formatted to match the paper's supplemental **eTable 3**.

### How to customize scenarios
You can add multiple rows to the table by adding dictionaries to the `scenarios_to_run` list. For each scenario, you can define a `name` and any of the following global variables to override:

*   `Starting_Dose_Level` (Default: 2)
*   `Highest_Possible_Dose_Level` (Default: 5)
*   `Lowest_Possible_Dose_Level` (Default: 1)
*   `Waitlist_Time` (Default: 0)
*   `Course_Length` (Default: 28)
*   `Screen_Failure_Prob` (Default: 0.30)
*   `Inevaluable_Prob` (Default: 0.20)
*   `dlt_prob` (defaults to arctan formula)
    * To use, provide a list that contains values for every dose level desired.
    * Examlple:  "dlt_probs": [0.10, 0.15, 0.25, 0.30, 0.35, 0.50] for 6 dose levels.  

### Function Parameters
1. **`scenarios`**: The list of dictionaries defined above.
2. **`sim_types`**: The simulation IDs to run side-by-side (e.g., `[1, 2]` for 3+3 and IQ 3+3).
3. **`num_repetitions`**: The number of simulated trials per scenario (e.g., 800).
4. **`seed`**: Change to random integer to change the patient batch. (default: 0)
    


###Example 1

In [51]:
# Define your scenarios here.
# Anything not defined in the dict will use the current global default.
scenarios_to_run = [
    {
        "name": "Scenario A1\nAve Ineval\n20% Ineval",
        "Inevaluable_Prob": 0.20,
        "Starting_Dose_Level": 2
    },
    {
        "name": "Scenario B1\nAve Ineval\n20% Ineval",
        "Inevaluable_Prob": 0.20,
        "Starting_Dose_Level": 2,
        "dlt_probs": [0.10, 0.15, 0.25, 0.30, 0.35]
    },
    {
        "name": "Scenario A2\nLow Ineval\n3.6% Ineval",
        "Inevaluable_Prob": 0.036,
        "Starting_Dose_Level": 2
    },
    {
        "name": "Scenario B2\nLow Ineval\n3.6% Ineval",
        "Inevaluable_Prob": 0.036,
        "Starting_Dose_Level": 2,
        "dlt_probs": [0.10, 0.15, 0.25, 0.30, 0.35, 0.50]
    }
]

print("Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)")
run_batch_report(scenarios_to_run, [1, 2], 8000, seed = 165198498432198)

Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)
+-------------+-------------+-------------+--------------+--------------+-------------+-------------+--------------+------------+------------+-------------+
| Variation   | #Treated    | #Treated    | #Evaluable   | #Evaluable   | #Months     | #Months     | Mean         | 3+3        | IQ 3+3     | DLT rate    |
|             | 3+3         | IQ 3+3      | 3+3          | IQ 3+3       | 3+3         | IQ 3+3      | #DLTs        | % MTD      | % MTD      | (based      |
|             | Mean, Med   | Mean, Med   | Mean, Med    | Mean, Med    | Mean, Med   | Mean, Med   | above        | At level   | At level   | on model)   |
|             | (range)     | (range)     | (range)      | (range)      | (range)     | (range)     | MTD          |            |            |             |
+=============+=============+=============+==============+==============+=============+=============+==============+============+============+==========

###Example 2

In [53]:
scenarios_to_run = [
    {
        "name": "Example \nScenario 1\n(ineval:0)",
        "Inevaluable_Prob": 0,
        "Screen_Failure_Prob": 0,
        "Starting_Dose_Level": 1,
        "Highest_Possible_Dose_Level": 1,
        "Lowest_Possible_Dose_Level": 1
    },
    {
        "name": "Example \nScenario 2\n(ineval:0.5)",
        "Inevaluable_Prob": 0.5,
        "Screen_Failure_Prob": 0,
        "Starting_Dose_Level": 1,
        "Highest_Possible_Dose_Level": 1,
        "Lowest_Possible_Dose_Level": 1
    }

]

print("Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)")
run_batch_report(scenarios_to_run, [1, 2], 800)

Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)
+--------------+-------------+-------------+--------------+--------------+-------------+-------------+--------------+------------+------------+-------------+
| Variation    | #Treated    | #Treated    | #Evaluable   | #Evaluable   | #Months     | #Months     | Mean         | 3+3        | IQ 3+3     | DLT rate    |
|              | 3+3         | IQ 3+3      | 3+3          | IQ 3+3       | 3+3         | IQ 3+3      | #DLTs        | % MTD      | % MTD      | (based      |
|              | Mean, Med   | Mean, Med   | Mean, Med    | Mean, Med    | Mean, Med   | Mean, Med   | above        | At level   | At level   | on model)   |
|              | (range)     | (range)     | (range)      | (range)      | (range)     | (range)     | MTD          |            |            |             |
+==============+=============+=============+==============+==============+=============+=============+==============+============+============+====

###Example 3

In [54]:
# Define your scenarios here.
# Anything not defined in the dict will use the current global default.
scenarios_to_run = [
    {
        "name": "Scenario A1\nAve Ineval\n20% Ineval",
        "Inevaluable_Prob": 0.20,
        "Starting_Dose_Level": 2,

    },
    {
        "name": "Scenario A2\nLow Ineval\n3.6% Ineval",
        "Inevaluable_Prob": 0.036,
        "Starting_Dose_Level": 2,

    }
]

print("Simulations for Rolling 6 and IQ Rolling 6 (each based on 800 simulations)")
run_batch_report(scenarios_to_run, [3, 4], 800)

Simulations for Rolling 6 and IQ Rolling 6 (each based on 800 simulations)
+-------------+-------------+----------------+--------------+----------------+-------------+----------------+--------------------+-------------+----------------+-------------+
| Variation   | #Treated    | #Treated       | #Evaluable   | #Evaluable     | #Months     | #Months        | Mean               | Rolling 6   | IQ Rolling 6   | DLT rate    |
|             | Rolling 6   | IQ Rolling 6   | Rolling 6    | IQ Rolling 6   | Rolling 6   | IQ Rolling 6   | #DLTs              | % MTD       | % MTD          | (based      |
|             | Mean, Med   | Mean, Med      | Mean, Med    | Mean, Med      | Mean, Med   | Mean, Med      | above              | At level    | At level       | on model)   |
|             | (range)     | (range)        | (range)      | (range)        | (range)     | (range)        | MTD                |             |                |             |
+=============+=============+================

# Individual Simulation Event Log

Use this block to run a **single trial** and view the detailed step-by-step event logs. This is useful for debugging patient flow, dose escalations, and timing logic.

### How to use:
Call `run_debug_test()` and type the values you want to change. If you don't type a variable, it uses the default.

### Function Parameters
*   `sim_type`: 1=3+3, 2=IQ 3+3, 3=Rolling 6, 4=IQ Rolling 6
*   `sdl`: Starting Dose Level (Default: 2)
*   `hpd`: Highest Possible Dose (Default: 5)
*   `lpd`: Lowest Possible Dose (Default: 1)
*   `wt`: Waitlist Time (Default: 0)
*   `cl`: Course Length (Default: 28)
*   `sfp`: Screen Failure Prob (Default: 0.30)
*   `ip`: Inevaluable Prob (Default: 0.20)
*   `seed`: Integer value, (Default is random)


### Sample Generation
`run_debug_test_2(sim_type=1)`
```
Running simulation with seed: 123456
Beginning 3+3 Simulation
Day: 8    | Event: Stay       | Adding new patient 1 to screening.                                | States: {'treated': 1, 'DLT': 0, 'pending': 1, 'AboveLevelState': 'open'}
...
Day: 1073 | Event: MTD        | Moving patients [] to treated from pending, dose is MTD.          | States: {'treated': 6, 'DLT': 1, 'pending': 0, 'AboveLevelState': 'closed'}
Number of patients consented to screening: 42
Number of patients starting treatment: 31
Number of patients fully evaluable: 27
Final Dose Level: 4
Total Time Taken: 1074 days or ~35.36 months
```



###Run
- If you come across a case that is incorrect in the future you can now refer to it as the `seed` that it uses, which should be a lot easier to descriabe to each other.
- even if you leave `seed` blank, it will tell you what seed was used.

In [55]:
# here is an example of a trial that goes up and then back down
run_debug_test(sim_type=2, seed = 301486315800757557521424412977664080381)


Seed: 301486315800757557521424412977664080381
 Day: 29   | Event: Stay       | Adding new patient 1 to screening.                                            | Dose: 2    | States: {'treated': 1, 'DLT': 0, 'pending': 1, 'AboveLevelState': 'open'} 
Day: 46   | Event: Stay       | Patient 1 starts treatment, at dose level 2.                                  | Dose: 2    | States: {'treated': 1, 'DLT': 0, 'pending': 1, 'AboveLevelState': 'open'} 
Day: 52   | Event: Stay       | Adding new patient 2 to screening.                                            | Dose: 2    | States: {'treated': 2, 'DLT': 0, 'pending': 2, 'AboveLevelState': 'open'} 
Day: 54   | Event: Stay       | Adding new patient 3 to screening.                                            | Dose: 2    | States: {'treated': 3, 'DLT': 0, 'pending': 3, 'AboveLevelState': 'open'} 
Day: 74   | Event: Suspend    | Patient 1 passes treatment.                                                   | Dose: 2    | States: {'treated': 3, 'DLT'

In [70]:
# here is an example of a trial that goes up and then back down
run_debug_test(sim_type=2, seed = 187465165)


Seed: 187465165
 Day: 22.07  | Event: Stay       | Adding new patient 1 to screening.                                            | Dose: 2    | States: {'treated': 1, 'DLT': 0, 'pending': 1, 'AboveLevelState': 'open'} 
Day: 32.22  | Event: Stay       | Adding new patient 2 to screening.                                            | Dose: 2    | States: {'treated': 2, 'DLT': 0, 'pending': 2, 'AboveLevelState': 'open'} 
Day: 34.68  | Event: Stay       | Adding new patient 3 to screening.                                            | Dose: 2    | States: {'treated': 3, 'DLT': 0, 'pending': 3, 'AboveLevelState': 'open'} 
Day: 41.07  | Event: Suspend    | Patient 1 starts treatment, at dose level 2.                                  | Dose: 2    | States: {'treated': 3, 'DLT': 0, 'pending': 3, 'AboveLevelState': 'open'} 
Day: 42.68  | Event: Suspend    | Patient 3 failed screening.                                                   | Dose: 2    | States: {'treated': 2, 'DLT': 0, 'pending': 2, '

#end on error cases
- none at moment :)